In [1]:
import glob


import numpy as np
import pandas as pd

import networkx as nx
from matplotlib import pyplot as plt

from scipy.sparse import diags
from scipy.sparse.linalg import cg


from src.mwnetwork.clustering import load_core_graph
from utils.path import find_root

/anaconda3/envs/metaboenv/lib/python3.14/site-packages/sspa/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


In [2]:
ROOT = find_root()
CORE_GRAPH_DIR = ROOT / "checkpoints" / "metabolomics_workbench" / "core_graph"
NAFLD_ANALYSIS_DIR = ROOT / "checkpoints" / "metabolomics_workbench" / "nafld_analysis"

In [3]:
study_files = sorted(glob.glob(str(NAFLD_ANALYSIS_DIR / "*.csv")))
study_files

['/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST000916_nafld_severity.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST000977_nafld_vs_healthy.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST001710_nafld_severity.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST001711_nafld_severity.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST001842_nafld_vs_healthy.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST001843_nafld_vs_healthy.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/checkpoints/metabolomics_workbench/nafld_analysis/ST001964_nafld_severity.csv',
 '/home/saldan/Desktop/projects/mw-metabolite-network/ch

In [4]:
all_associations = pd.concat([
    pd.read_csv(f).dropna(subset=["kendall_tau"]).assign(study_id=f.split("/")[-1].split("_")[0]).set_index(["study_id", "refmet_id"])
    for f in study_files]
)
all_associations

kendall_tau   p_value    n
study_id refmet_id                            
ST000916 RM0001239    -0.106929  0.185349   88
         RM0001442    -0.165306  0.041830   87
         RM0001603     0.201491  0.012591   88
         RM0001619    -0.348049  0.000016   88
         RM0001782    -0.181874  0.024596   88
...                         ...       ...  ...
ST002269 RM0153846     0.155488  0.015631  165
         RM0154019     0.091550  0.209707  165
         RM0154093     0.081550  0.203134  165
         RM0154083     0.195892  0.002213  165
         RM0154023     0.028142  0.697953  165

[2092 rows x 3 columns]

In [5]:
def associations_purged_from(study_id):
    """
    Purge associations from a given study_id from the associations dataframe.
    """
    return all_associations.drop(study_id, level="study_id")

associations_purged_from("ST002269")

kendall_tau   p_value   n
study_id refmet_id                           
ST000916 RM0001239    -0.106929  0.185349  88
         RM0001442    -0.165306  0.041830  87
         RM0001603     0.201491  0.012591  88
         RM0001619    -0.348049  0.000016  88
         RM0001782    -0.181874  0.024596  88
...                         ...       ...  ..
ST002091 RM0071606     0.250780  0.029807  52
         RM0071607     0.237542  0.030957  57
         RM0055968    -0.005981  0.973808  22
         RM0056115     0.121306  0.195252  78
         RM0023722     0.102552  0.399467  47

[1750 rows x 3 columns]

In [6]:
def metabolite_association(refmet_id, associations=all_associations):
    """
    Calculate the overall association of a metabolite with its associated signals.
    """
    try:
        tau, n = associations.loc[(slice(None), refmet_id), ["kendall_tau", "n"]].values.T
        w = (n-4)/0.437
        z = np.arctanh(tau)
        z = (w*z).sum()/w.sum()
        return z
    except KeyError:
        return np.nan # metabolite not tested in any study in the associations dataframe

metabolite_association("RM0001239", associations=associations_purged_from("ST000916"))

np.float64(0.1502665376846832)

In [7]:
embeddings = pd.read_csv(CORE_GRAPH_DIR / "embedding_posonly.csv", index_col=0)
embeddings = embeddings.apply(np.array, axis=1).to_dict()
embeddings

{'RM0000021': array([ -9.128403, -23.010183]),
 'RM0000392': array([ -8.373507, -23.362133]),
 'RM0090135': array([ -8.059506, -24.006842]),
 'RM0097845': array([ -7.9872026, -22.681086 ]),
 'RM0133233': array([-22.339285 ,  -4.0641565]),
 'RM0134908': array([-10.399134, -11.173418]),
 'RM0135489': array([-15.816205, -26.544724]),
 'RM0135505': array([-17.103632, -28.52467 ]),
 'RM0138805': array([ 1.0584117, -9.715063 ]),
 'RM0000022': array([-11.490021, -26.893806]),
 'RM0000033': array([-11.200818, -26.808294]),
 'RM0000052': array([-12.045052, -26.989618]),
 'RM0000544': array([ -8.571793, -24.507908]),
 'RM0013838': array([-12.870199, -21.693346]),
 'RM0090012': array([-24.328062, -24.391743]),
 'RM0134064': array([-15.400362, -20.855791]),
 'RM0134083': array([-14.312689, -20.598171]),
 'RM0135485': array([-14.17411 , -29.588938]),
 'RM0135487': array([-16.254742, -30.221056]),
 'RM0160764': array([-20.504086, -25.744362]),
 'RM0171045': array([ -9.927366, -28.850132]),
 'RM00004

In [8]:
G, dist = load_core_graph(CORE_GRAPH_DIR, )

# select only nodes that are in the distance matrix (i.e. nodes that are not isolated after removing negative edges)
G = nx.subgraph(G, dist.index)

# remove negative edges
G = G.edge_subgraph([(u,v) for u,v,r in G.edges(data="r") if r>0])

dist

,RM0000021,RM0000392,RM0090135,RM0097845,RM0133233,RM0134908,RM0135489,RM0135505,RM0138805,RM0000022,...,RM0153782,RM0153079,RM0153081,RM0153083,RM0153147,RM0161911,RM0153242,RM0224203,RM0153784,RM0154174
RM0000021,0.000000,0.378429,0.470981,0.418583,0.743224,0.466469,0.720599,0.517737,0.548854,1.276542,...,1.937694,1.643785,1.704059,1.456993,2.666137,2.185107,1.886859,1.818476,1.640611,1.405595
RM0000392,0.378429,0.000000,0.261451,0.221600,0.612044,0.328705,0.399774,0.497072,0.625782,1.229385,...,1.740710,1.535906,1.596180,1.349114,2.414051,1.988124,1.603717,1.535334,1.318003,1.176224
RM0090135,0.470981,0.261451,0.000000,0.441875,0.587624,0.553591,0.546043,0.571209,0.571522,1.086001,...,1.649824,1.507348,1.567622,1.320556,2.544769,2.047378,1.593003,1.533845,1.474753,1.132345
RM0097845,0.418583,0.221600,0.441875,0.000000,0.561150,0.188085,0.367810,0.521535,0.538548,1.103893,...,1.519111,1.400465,1.450494,1.213673,2.278610,1.766524,1.468276,1.399893,1.222028,0.996132
RM0133233,0.743224,0.612044,0.587624,0.561150,0.000000,0.566448,0.554518,0.510065,0.522295,1.067293,...,1.378009,1.386929,1.447203,1.200137,2.162928,1.766322,1.445356,1.376973,0.907539,0.973509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RM0161911,2.185107,1.988124,2.047378,1.766524,1.766322,1.917454,1.923068,1.955272,1.878724,2.516613,...,2.816782,2.193620,2.113322,2.006828,3.458029,0.000000,2.519219,2.505553,2.511956,2.243189
RM0153242,1.886859,1.603717,1.593003,1.468276,1.445356,1.458378,1.523541,1.502974,1.476979,1.970440,...,2.203899,1.872093,1.932368,1.685302,2.736418,2.519219,0.000000,1.854629,1.972182,1.465434
RM0224203,1.818476,1.535334,1.533845,1.399893,1.376973,1.389995,1.464382,1.439166,1.408596,1.902057,...,2.138452,1.795483,1.855758,1.608692,2.797170,2.505553,1.854629,0.000000,1.901098,1.592447
RM0153784,1.640611,1.318003,1.474753,1.222028,0.907539,1.313361,1.174238,1.394534,1.207746,1.947723,...,1.584094,1.709964,1.740108,1.507245,2.488834,2.511956,1.972182,1.901098,0.000000,1.525166


In [9]:
# G number of connected components
len([None for component in nx.connected_components(G)])

1

In [10]:
SUPER_CLASSES = set([G.nodes()[node]["super_class"] for node in G.nodes() if G.nodes[node]["super_class"]])
SUPER_CLASS_ENCODING = {super_class: i for i, super_class in enumerate(SUPER_CLASSES)}
SUPER_CLASS_ENCODING

{'Organic oxygen compounds': 0,
 'Alkaloids': 1,
 'Prenol Lipids': 2,
 'Organic acids': 3,
 'Sterol Lipids': 4,
 'Carbohydrates': 5,
 'Glycerophospholipids': 6,
 'Inorganic compounds': 7,
 'Organosulfur compounds': 8,
 'Lignans': 9,
 'Glycerolipids': 10,
 'Organic nitrogen compounds': 11,
 'Benzenoids': 12,
 'Fatty Acyls': 13,
 'Nucleic acids': 14,
 'Organohalogen compounds': 15,
 'Organoheterocyclic compounds': 16,
 'Sphingolipids': 17,
 'Polyketides': 18}

In [11]:
G.nodes()["RM0001239"]

{'refmet_name': 'PE 34:2',
 'super_class': 'Glycerophospholipids',
 'main_class': 'Glycerophosphoethanolamines',
 'sub_class': 'PE',
 'formula': 'C39H74NO8P',
 'exactmass': 715.515207,
 'pubchem_cid': nan,
 'chebi_id': 71721.0,
 'hmdb_id': nan,
 'lipidmaps_id': nan,
 'kegg_id': nan,
 'inchi_key': nan}

In [12]:
def annotate_metabolites(refmet_ids, associations=all_associations):
    """
    Return the attributes of a node in the graph.
    """
    z = np.array([metabolite_association(refmet_id, associations=associations) for refmet_id in refmet_ids])
    return {
        "z": z,
        "super_class": np.array([G.nodes()[refmet_id]["super_class"] for refmet_id in refmet_ids]),
        "exactmass": np.array([G.nodes()[refmet_id]["exactmass"] for refmet_id in refmet_ids]),
        
    }
annotate_metabolites(["RM0001239", "RM0097845", "RM0154019"])

{'z': array([0.11712889,        nan, 0.09180745]),
 'super_class': array(['Glycerophospholipids', 'Glycerophospholipids', 'Fatty Acyls'],
       dtype='<U20'),
 'exactmass': array([715.515207, 785.593457, 275.173274])}

In [37]:
def get_metabolites_features(refmet_ids, associations=all_associations):
    """
    Return numerical representation of a metabolite's attributes
    """
    attrs = annotate_metabolites(refmet_ids, associations=associations)
    super_class_ohe = np.zeros((len(refmet_ids), len(SUPER_CLASSES)))
    for i, super_class in enumerate(attrs["super_class"]):
        super_class_ohe[i, SUPER_CLASS_ENCODING[super_class]] = 1

    return np.concatenate([
        super_class_ohe,
        attrs["exactmass"].reshape(-1, 1),
        attrs["z"].reshape(-1, 1)
    ], axis=1)

get_metabolites_features(["RM0001239", "RM0097845", "RM0154019"], associations=associations_purged_from("ST000916"))

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 7.15515207e+02,
        1.50266538e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 7.85593457e+02,
                   nan],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.0000

In [36]:
def get_graph_features(associations=all_associations):
    return get_metabolites_features(np.array(G.nodes()), associations=associations)

In [44]:
def propagate(G, y, precision, alpha, nodelist):
    W = nx.to_scipy_sparse_array(G, nodelist=nodelist, weight="weight", format="csr")
    L = diags(W.sum(axis=1).A1) - W          # sparse Laplacian
    S = diags(precision)                      # 0 for untested, 1/SE^2 for tested
    A = S + alpha * L
    f, info = cg(A, S @ y)                    # conjugate gradient, not explicit inverse
    return f



/tmp/ipykernel_2145449/1779235639.py:9: RuntimeWarning: invalid value encountered in scalar divide
  z = (w*z).sum()/w.sum()
/tmp/ipykernel_2145449/1779235639.py:9: RuntimeWarning: invalid value encountered in scalar divide
  z = (w*z).sum()/w.sum()


AttributeError: 'numpy.ndarray' object has no attribute 'A1'

In [45]:
nx.to_scipy_sparse_array(G, weight="weight", format="csr")

<Compressed Sparse Row sparse array of dtype 'float64'
	with 1586130 stored elements and shape (4429, 4429)>